In [1]:
import pandas as pd
import numpy as np
import re
import os

CLEAN_DIR = "data/cleaned"
READY_DIR = "data/recommendation_ready"

os.makedirs(READY_DIR, exist_ok=True)

people = pd.read_csv(f"{CLEAN_DIR}/people_cleaned.csv")
education = pd.read_csv(f"{CLEAN_DIR}/education_cleaned.csv")
experience = pd.read_csv(f"{CLEAN_DIR}/experience_cleaned.csv")
person_skills = pd.read_csv(f"{CLEAN_DIR}/person_skills_cleaned.csv")
courses = pd.read_csv(f"{CLEAN_DIR}/courses_cleaned.csv")

In [2]:
def normalize_skill(skill):
    if pd.isna(skill):
        return None
    
    skill = str(skill).lower().strip()
    
    # Normalize common separators
    skill = skill.replace("&", " and ")
    skill = skill.replace("/", " ")
    skill = skill.replace("-", " ")
    
    # Remove punctuation
    skill = re.sub(r"[^a-z0-9+#.\s]", " ", skill)
    
    # Collapse whitespace
    skill = re.sub(r"\s+", " ", skill).strip()
    
    return skill if skill else None

In [3]:
person_skills["skill_normalized"] = (
    person_skills["skill"]
    .apply(normalize_skill)
)

person_skills = person_skills.dropna(
    subset=["skill_normalized"]
)

person_skills = person_skills.drop_duplicates(
    subset=["person_id", "skill_normalized"]
)

In [4]:
user_skills = (
    person_skills
    .groupby("person_id")["skill_normalized"]
    .agg(list)
    .reset_index()
    .rename(columns={
        "skill_normalized": "skills"
    })
)

In [5]:
user_skills["skills_text"] = user_skills["skills"].apply(
    lambda x: " ".join(x)
)

In [6]:
user_skills["skill_count"] = (
    user_skills["skills"].apply(len)
)

In [7]:
user_skills.head()

,person_id,skills,skills_text,skill_count
0,1,"[database administration, database, ms sql ser...",database administration database ms sql server...,20
1,2,"[sql server management studio, visual studio, ...",sql server management studio visual studio sql...,17
2,3,"[databases, oracle 4 years, oracle 10g, sql, l...",databases oracle 4 years oracle 10g sql linux ...,7
3,4,[maintain multiple database environments redsh...,maintain multiple database environments redshi...,26
4,5,"[scrum, agile software development, product ba...",scrum agile software development product backl...,27


In [8]:
experience["title_normalized"] = (
    experience["title"]
    .apply(normalize_skill)
)

In [9]:
user_experience = (
    experience
    .dropna(subset=["title_normalized"])
    .groupby("person_id")
    .agg(
        experience_titles=("title_normalized", list),
        experience_count=("title_normalized", "size")
    )
    .reset_index()
)

In [10]:
user_experience["experience_text"] = (
    user_experience["experience_titles"]
    .apply(lambda x: " ".join(x))
)

In [11]:
user_experience.head()

,person_id,experience_titles,experience_count,experience_text
0,1,"[database administrator, database administrator]",2,database administrator database administrator
1,2,[database administrator],1,database administrator
2,3,"[oracle database administrator, oracle databas...",2,oracle database administrator oracle database ...
3,4,[amazon redshift administrator and etl develop...,3,amazon redshift administrator and etl develope...
4,5,"[scrum master, oracle database administrator s...",3,scrum master oracle database administrator scr...


In [12]:
user_experience = (
    experience
    .dropna(subset=["title_normalized"])
    .groupby("person_id")
    .agg(
        experience_titles=(
            "title_normalized",
            lambda x: list(dict.fromkeys(x))
        ),
        experience_count=("title_normalized", "size")
    )
    .reset_index()
)

user_experience["experience_text"] = (
    user_experience["experience_titles"]
    .apply(lambda x: " ".join(x))
)

In [13]:
education["program_normalized"] = (
    education["program"]
    .apply(normalize_skill)
)

In [14]:
user_education = (
    education
    .dropna(subset=["program_normalized"])
    .groupby("person_id")
    .agg(
        education_programs=(
            "program_normalized",
            lambda x: list(dict.fromkeys(x))
        ),
        education_count=("program_normalized", "size")
    )
    .reset_index()
)

In [15]:
user_education["education_text"] = (
    user_education["education_programs"]
    .apply(lambda x: " ".join(x))
)

In [16]:
user_profiles = people.copy()

user_profiles = user_profiles.merge(
    user_skills[
        ["person_id", "skills", "skills_text", "skill_count"]
    ],
    on="person_id",
    how="left"
)

user_profiles = user_profiles.merge(
    user_experience[
        [
            "person_id",
            "experience_titles",
            "experience_text",
            "experience_count"
        ]
    ],
    on="person_id",
    how="left"
)

user_profiles = user_profiles.merge(
    user_education[
        [
            "person_id",
            "education_programs",
            "education_text",
            "education_count"
        ]
    ],
    on="person_id",
    how="left"
)

In [17]:
user_profiles["skills"] = (
    user_profiles["skills"]
    .apply(lambda x: x if isinstance(x, list) else [])
)

user_profiles["experience_titles"] = (
    user_profiles["experience_titles"]
    .apply(lambda x: x if isinstance(x, list) else [])
)

user_profiles["education_programs"] = (
    user_profiles["education_programs"]
    .apply(lambda x: x if isinstance(x, list) else [])
)

for col in [
    "skills_text",
    "experience_text",
    "education_text"
]:
    user_profiles[col] = user_profiles[col].fillna("")
    
for col in [
    "skill_count",
    "experience_count",
    "education_count"
]:
    user_profiles[col] = user_profiles[col].fillna(0).astype(int)

In [18]:
user_profiles["profile_text"] = (
    "skills " + user_profiles["skills_text"] +
    " experience " + user_profiles["experience_text"] +
    " education " + user_profiles["education_text"]
)

In [19]:
user_profiles

,person_id,name,skills,skills_text,skill_count,experience_titles,experience_text,experience_count,education_programs,education_text,education_count,profile_text
0,1,Database Administrator,"[database administration, database, ms sql ser...",database administration database ms sql server...,20,[database administrator],database administrator,2,[bachelor of science],bachelor of science,1,skills database administration database ms sql...
1,2,Database Administrator,"[sql server management studio, visual studio, ...",sql server management studio visual studio sql...,17,[database administrator],database administrator,1,[bsc in computer science],bsc in computer science,1,skills sql server management studio visual stu...
2,3,Oracle Database Administrator,"[databases, oracle 4 years, oracle 10g, sql, l...",databases oracle 4 years oracle 10g sql linux ...,7,[oracle database administrator],oracle database administrator,2,[master of computer applications in science an...,master of computer applications in science and...,1,skills databases oracle 4 years oracle 10g sql...
3,4,Amazon Redshift Administrator and ETL Develope...,[maintain multiple database environments redsh...,maintain multiple database environments redshi...,26,[amazon redshift administrator and etl develop...,amazon redshift administrator and etl develope...,3,[bachelor in computer science],bachelor in computer science,1,skills maintain multiple database environments...
4,5,Scrum Master Scrum Master Scrum Master,"[scrum, agile software development, product ba...",scrum agile software development product backl...,27,"[scrum master, oracle database administrator s...",scrum master oracle database administrator scr...,3,[],,0,skills scrum agile software development produc...
...,...,...,...,...,...,...,...,...,...,...,...,...
54928,54929,Lead Python Developer,"[django, angular js, javascript, jquery, node....",django angular js javascript jquery node.js py...,82,"[lead python developer, sr. python developer, ...",lead python developer sr. python developer pyt...,6,[],,0,skills django angular js javascript jquery nod...
54929,54930,Full Stack Python Developer,"[python, django, aws, angularjs, bootstrap, ja...",python django aws angularjs bootstrap javascri...,42,"[full stack python developer, sr. python devel...",full stack python developer sr. python develop...,6,[],,0,skills python django aws angularjs bootstrap j...
54930,54931,Eli Lilly,"[python 2.7, html5, css3, ajax, json, jquery, ...",python 2.7 html5 css3 ajax json jquery active ...,110,"[sr. python developer, python developer, java ...",sr. python developer python developer java dev...,6,[],,0,skills python 2.7 html5 css3 ajax json jquery ...
54931,54932,Python Developer,"[python 3.1x, pyquery, pyqt, django, angular.j...",python 3.1x pyquery pyqt django angular.js ope...,47,"[python developer, software developer]",python developer software developer,6,[],,0,skills python 3.1x pyquery pyqt django angular...


In [20]:
user_profiles["career_context"] = (
    user_profiles["experience_titles"]
    .apply(lambda x: " | ".join(x))
)

In [21]:
if "unnamed_0" in courses.columns:
    courses = courses.drop(columns=["unnamed_0"])

In [22]:
courses = courses.reset_index(drop=True)

courses["course_id"] = (
    "COURSE_" +
    courses.index.astype(str).str.zfill(4)
)

In [23]:
def parse_course_skills(value):
    if pd.isna(value):
        return []
    
    skills = str(value).split(",")
    
    skills = [
        normalize_skill(skill)
        for skill in skills
    ]
    
    skills = [
        skill for skill in skills
        if skill
    ]
    
    # Remove duplicates while preserving order
    return list(dict.fromkeys(skills))

In [24]:
courses["skills_list"] = (
    courses["skills"]
    .apply(parse_course_skills)
)

In [25]:
courses["skills_normalized"] = (
    courses["skills_list"]
    .apply(lambda x: " ".join(x))
)

In [26]:
courses["skill_count"] = (
    courses["skills_list"].apply(len)
)

In [27]:
courses[
    ["course_id", "title", "skills", "skills_list", "skill_count"]
].head(10)

,course_id,title,skills,skills_list,skill_count
0,COURSE_0000,Google Cybersecurity,"Network Security, Python Programming, Linux, C...","[network security, python programming, linux, ...",14
1,COURSE_0001,Google Data Analytics,"Data Analysis, R Programming, SQL, Business Co...","[data analysis, r programming, sql, business c...",25
2,COURSE_0002,Google Project Management:,"Project Management, Strategy and Operations, L...","[project management, strategy and operations, ...",24
3,COURSE_0003,IBM Data Science,"Python Programming, Data Science, Machine Lear...","[python programming, data science, machine lea...",32
4,COURSE_0004,Google Digital Marketing & E-commerce,"Digital Marketing, Marketing, Marketing Manage...","[digital marketing, marketing, marketing manag...",19
5,COURSE_0005,IBM Data Analyst,"Python Programming, Microsoft Excel, Data Visu...","[python programming, microsoft excel, data vis...",31
6,COURSE_0006,Google IT Support,"Computer Networking, Network Architecture, Net...","[computer networking, network architecture, ne...",19
7,COURSE_0007,Machine Learning,"Machine Learning, Machine Learning Algorithms,...","[machine learning, machine learning algorithms...",16
8,COURSE_0008,Google UX Design,"User Experience, User Experience Design, User ...","[user experience, user experience design, user...",12
9,COURSE_0009,IBM DevOps and Software Engineering,"DevOps, Software Engineering, Cloud Computing,...","[devops, software engineering, cloud computing...",33


In [28]:
courses["title_normalized"] = (
    courses["title"]
    .apply(normalize_skill)
)

courses["organization_normalized"] = (
    courses["organization"]
    .apply(normalize_skill)
)

In [29]:
courses["course_description_clean"] = (
    courses["course_description"]
    .fillna("")
    .astype(str)
)

In [30]:
courses["course_text"] = (
    "title " + courses["title_normalized"].fillna("") +
    " skills " + courses["skills_normalized"].fillna("") +
    " organization " + courses["organization_normalized"].fillna("") +
    " description " + courses["course_description_clean"]
)

In [31]:
courses["skill_text"] = (
    courses["title_normalized"].fillna("") +
    " " +
    courses["skills_normalized"].fillna("")
)

In [32]:
courses["content_text"] = (
    courses["title_normalized"].fillna("") +
    " " +
    courses["skills_normalized"].fillna("") +
    " " +
    courses["course_description_clean"]
)

In [33]:
courses["ratings"] = pd.to_numeric(
    courses["ratings"],
    errors="coerce"
)
courses["review_count"] = pd.to_numeric(
    courses["review_count"],
    errors="coerce"
)
courses["course_students_enrolled"] = pd.to_numeric(
    courses["course_students_enrolled"],
    errors="coerce"
)

In [34]:
print(courses["difficulty"].value_counts(dropna=False))

difficulty
Beginner        295
Intermediate     76
Mixed            18
Advanced         15
Name: count, dtype: int64


In [35]:
courses["difficulty_normalized"] = (
    courses["difficulty"]
    .fillna("unknown")
    .astype(str)
    .str.lower()
    .str.strip()
)

In [36]:
courses["recommendation_text"] = (
    "title " +
    courses["title_normalized"].fillna("") +
    " skills " +
    courses["skills_normalized"].fillna("")
)

In [37]:
user_profiles.to_csv(
    f"{READY_DIR}/user_profiles.csv",
    index=False
)
courses.to_csv(
    f"{READY_DIR}/course_profiles.csv",
    index=False
)


In [38]:
print("USER PROFILES")
print("=" * 50)
print(user_profiles.shape)
print(user_profiles.columns.tolist())
display(user_profiles.head(3))

USER PROFILES
(54933, 13)
['person_id', 'name', 'skills', 'skills_text', 'skill_count', 'experience_titles', 'experience_text', 'experience_count', 'education_programs', 'education_text', 'education_count', 'profile_text', 'career_context']


,person_id,name,skills,skills_text,skill_count,experience_titles,experience_text,experience_count,education_programs,education_text,education_count,profile_text,career_context
0,1,Database Administrator,"[database administration, database, ms sql ser...",database administration database ms sql server...,20,[database administrator],database administrator,2,[bachelor of science],bachelor of science,1,skills database administration database ms sql...,database administrator
1,2,Database Administrator,"[sql server management studio, visual studio, ...",sql server management studio visual studio sql...,17,[database administrator],database administrator,1,[bsc in computer science],bsc in computer science,1,skills sql server management studio visual stu...,database administrator
2,3,Oracle Database Administrator,"[databases, oracle 4 years, oracle 10g, sql, l...",databases oracle 4 years oracle 10g sql linux ...,7,[oracle database administrator],oracle database administrator,2,[master of computer applications in science an...,master of computer applications in science and...,1,skills databases oracle 4 years oracle 10g sql...,oracle database administrator


In [39]:
print("\nCOURSE PROFILES")
print("=" * 50)
print(courses.shape)
print(courses.columns.tolist())
display(
    courses[
        [
            "course_id",
            "title",
            "skills",
            "skills_normalized",
            "skill_count",
            "difficulty_normalized"
        ]
    ].head(5)
)


COURSE PROFILES
(404, 23)
['title', 'organization', 'skills', 'ratings', 'course_url', 'course_students_enrolled', 'course_description', 'review_count', 'difficulty', 'type', 'duration', 'skills_normalized', 'course_id', 'skills_list', 'skill_count', 'title_normalized', 'organization_normalized', 'course_description_clean', 'course_text', 'skill_text', 'content_text', 'difficulty_normalized', 'recommendation_text']


,course_id,title,skills,skills_normalized,skill_count,difficulty_normalized
0,COURSE_0000,Google Cybersecurity,"Network Security, Python Programming, Linux, C...",network security python programming linux clou...,14,beginner
1,COURSE_0001,Google Data Analytics,"Data Analysis, R Programming, SQL, Business Co...",data analysis r programming sql business commu...,25,beginner
2,COURSE_0002,Google Project Management:,"Project Management, Strategy and Operations, L...",project management strategy and operations lea...,24,beginner
3,COURSE_0003,IBM Data Science,"Python Programming, Data Science, Machine Lear...",python programming data science machine learni...,32,beginner
4,COURSE_0004,Google Digital Marketing & E-commerce,"Digital Marketing, Marketing, Marketing Manage...",digital marketing marketing marketing manageme...,19,beginner


In [40]:
courses

,title,organization,skills,ratings,course_url,course_students_enrolled,course_description,review_count,difficulty,type,...,skills_list,skill_count,title_normalized,organization_normalized,course_description_clean,course_text,skill_text,content_text,difficulty_normalized,recommendation_text
0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, C...",4.8,https://www.coursera.org/professional-certific...,700909.0,Google Cloud Fundamentals: Core Infrastructure...,NaN,Beginner,Professional Certificate,...,"[network security, python programming, linux, ...",14,google cybersecurity,google,Google Cloud Fundamentals: Core Infrastructure...,title google cybersecurity skills network secu...,google cybersecurity network security python p...,google cybersecurity network security python p...,beginner,title google cybersecurity skills network secu...
1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business Co...",4.8,https://www.coursera.org/professional-certific...,229865.0,Prepare for a new career in the high-growth fi...,NaN,Beginner,Professional Certificate,...,"[data analysis, r programming, sql, business c...",25,google data analytics,google,Prepare for a new career in the high-growth fi...,title google data analytics skills data analys...,google data analytics data analysis r programm...,google data analytics data analysis r programm...,beginner,title google data analytics skills data analys...
2,Google Project Management:,Google,"Project Management, Strategy and Operations, L...",4.8,https://www.coursera.org/professional-certific...,29702.0,Prepare-se para uma nova carreira no campo de ...,NaN,Beginner,Professional Certificate,...,"[project management, strategy and operations, ...",24,google project management,google,Prepare-se para uma nova carreira no campo de ...,title google project management skills project...,google project management project management s...,google project management project management s...,beginner,title google project management skills project...
3,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lear...",4.6,https://www.coursera.org/professional-certific...,239622.0,Prepare for a career in the high-growth field ...,NaN,Beginner,Professional Certificate,...,"[python programming, data science, machine lea...",32,ibm data science,ibm,Prepare for a career in the high-growth field ...,title ibm data science skills python programmi...,ibm data science python programming data scien...,ibm data science python programming data scien...,beginner,title ibm data science skills python programmi...
4,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manage...",4.8,https://www.coursera.org/professional-certific...,384238.0,This course is the eighth course in the Google...,NaN,Beginner,Professional Certificate,...,"[digital marketing, marketing, marketing manag...",19,google digital marketing and e commerce,google,This course is the eighth course in the Google...,title google digital marketing and e commerce ...,google digital marketing and e commerce digita...,google digital marketing and e commerce digita...,beginner,title google digital marketing and e commerce ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399,Digital Manufacturing & Design Technology,University at Buffalo,"Design and Product, Leadership and Management,...",4.6,https://www.coursera.org/specializations/digit...,79172.0,Ce cours est conçu pour accompagner toutes les...,NaN,Beginner,Specialization,...,"[design and product, leadership and management...",44,digital manufacturing and design technology,university at buffalo,Ce cours est conçu pour accompagner toutes les...,title digital manufacturing and design technol...,digital manufacturing and design technology de...,digital manufacturing and design technology de...,beginner,title digital manufacturing and design technol...
400,Cybersecurity for Everyone,"University of Ma